## Plots for results

### Images

In [ ]:
import os # thesis-segmentation ENV
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib.cm as cm
import matplotlib.colors as colors
import textwrap
import pickle
from scipy.stats import wilcoxon, mannwhitneyu

from skimage.transform import resize, AffineTransform, warp

from tifffile import imread

from __future__ import print_function

In [ ]:
import sys
sys.path.insert(0, r"D:\Masters_Medical_Informatics_Larger_Files\Thesis\Working_Folder\Masters-Thesis\src\he2multi_reg")

In [ ]:
import importlib, he2multi_reg
importlib.reload(he2multi_reg)
importlib.reload(he2multi_reg.reg)
importlib.reload(he2multi_reg.metrics)
importlib.reload(he2multi_reg.regPipeline)
importlib.reload(he2multi_reg.preprocess)

In [ ]:
def load_image_data(folder_path):

    img_data = []
    label_data = []
    
    for _, _, files in os.walk(folder_path):
        for index, file in enumerate(files):
            if file.endswith(".tif"): 
                image_path = os.path.join(folder_path, file)
                img_raw = imread(image_path)
                img = np.array(img_raw)
                img_data.append(img)

    return img_data

#### cropped images

In [ ]:
hne_px_sz = 0.5023
dapi_px_sz = 0.209877

scale = hne_px_sz / dapi_px_sz

# load dapi
dapi_img_data = load_image_data("../../../Data/for_testing/exmp_1/cropped/dapi")
dapi1_init = resize(dapi_img_data[0], (int(dapi_img_data[0].shape[0]/scale), int(dapi_img_data[0].shape[1]/scale)), anti_aliasing=True)
dapi2_init = resize(dapi_img_data[1], (int(dapi_img_data[1].shape[0]/scale), int(dapi_img_data[1].shape[1]/scale)), anti_aliasing=True)
dapi1_init, dapi2_init = dapi1_init*255, dapi2_init*255

# load hne
hne_image_data = load_image_data("../../../Data/for_testing/exmp_1/cropped/hne")
hne1_init = hne_image_data[0]
hne2_init = hne_image_data[1]

hne1_deconv = he2multi_reg.colour_deconvolusion_preprocessing_HnE(hne1_init)
hne2_deconv = he2multi_reg.colour_deconvolusion_preprocessing_HnE(hne2_init)

In [ ]:
def overlay_images(dapi_img, hne_img, axis, alpha_dapi=1, alpha_hne=0.35):
    axis.imshow(dapi_img, cmap='Greens', alpha=alpha_dapi)
    axis.imshow(hne_img, cmap='Reds', alpha=alpha_hne)
    axis.set_aspect('equal')
    axis.set_xticks([])
    axis.set_yticks([])

def get_nice_ticks(max_val, n_ticks=5):
    step = max_val // n_ticks
    step = int(np.ceil(step / 50) * 50)  # round to nearest 50
    ticks = np.arange(0, step * n_ticks, step)  # exclude the last value to avoid overlap
    return ticks

def plot_comparison_grid(
        dapi_list,
        hne_list,
        deconv_list,
        registered_list_dict,
        column_titles=None
    ):
    n_rows = len(dapi_list)
    reg_methods = list(registered_list_dict.keys())
    n_cols = 3 + len(reg_methods)

    # set equal image sizes
    min_shape = np.min([img.shape[:2] for img in dapi_list], axis=0)
    def resize_to_min_shape(img_list, min_shape):
        return [img[:min_shape[0], :min_shape[1]] for img in img_list]

    dapi_list = resize_to_min_shape(dapi_list, min_shape)
    hne_list = resize_to_min_shape(hne_list, min_shape)
    deconv_list = resize_to_min_shape(deconv_list, min_shape)
    for method in reg_methods:
        registered_list_dict[method] = resize_to_min_shape(registered_list_dict[method], min_shape)

    if column_titles is None:
        column_titles = ["DAPI", "H&E", "initial overlay"] + reg_methods

    fig, axes = plt.subplots(
        n_rows + 1,
        n_cols,
        figsize=(1.5 * n_cols, 2.6 * (n_rows + 0.2)),
        squeeze=False
    )

    # top row titles
    for j in range(n_cols):
        ax = axes[0, j]
        title = column_titles[j]
        wrapped_title = "\n".join(textwrap.wrap(title, width=15))
        ax.set_title(wrapped_title, fontsize=12, pad=3)
        ax.axis("off")

    # image rows
    n_ticks = 5

    # image rows
    for i in range(n_rows):
        for j in range(n_cols):
            ax = axes[i + 1, j]
        
            # which image to plot
            if j == 0:
                img = dapi_list[i]
            elif j == 1:
                img = hne_list[i]
            elif j == 2:
                img = deconv_list[i]
            else:
                method = reg_methods[j-3]
                img = registered_list_dict[method][i]
        
            # overlay 
            if j >= 2:
                overlay_images(dapi_list[i], img, ax)
            else:
                ax.imshow(img, cmap="gray")
                ax.set_aspect('equal')
        
            # show only for left column (y) and bottom row (x)
            if j == 0:  # left column
                yticks = get_nice_ticks(min_shape[0], n_ticks=5)
                ax.set_yticks(yticks)
                ax.set_yticklabels([f"{int(y)}" for y in yticks], fontsize=8)
            else:
                ax.set_yticks([])

            if i == n_rows - 1:  # bottom row
                xticks = get_nice_ticks(min_shape[1], n_ticks=5)
                ax.set_xticks(xticks)
                ax.set_xticklabels([f"{int(x)}" for x in xticks], fontsize=8)
            else:
                ax.set_xticks([])


    plt.subplots_adjust(
        left=0.03, right=0.97,
        top=0.95, bottom=0.05,
        wspace=0.02,
        hspace=0.01
    )

    plt.show()

In [ ]:
img1_tforms, img1_registered, img1_tre_pts = he2multi_reg.register_DAPI_HnE(dapi1_init, hne1_deconv, adv_tform='intensity', intensity_tform='r-af-bs', mpp=hne_px_sz)

In [ ]:
img2_tforms, img2_registered, img2_tre_pts = he2multi_reg.register_DAPI_HnE(dapi2_init, hne2_deconv, adv_tform='intensity', intensity_tform='r-af-bs', mpp=hne_px_sz)

In [ ]:
hne1_applying_tform = AffineTransform(scale=(1.0, 1.0), rotation=0.5, translation=(100, -200))
hne1_testing_hne_img = warp(hne1_init, hne1_applying_tform)
hne1_testing_img = warp(hne1_deconv, hne1_applying_tform)
img1_test_tforms, img1_test_registered, img1_test_tre_pts = he2multi_reg.register_DAPI_HnE(dapi1_init, hne1_testing_img, adv_tform='intensity', intensity_tform='r-af-bs', mpp=hne_px_sz)

In [ ]:
plot_comparison_grid(
    dapi_list=[dapi1_init, dapi2_init, dapi1_init],
    hne_list=[hne1_init, hne2_init, hne1_testing_hne_img],
    deconv_list=[hne1_deconv, hne2_deconv, hne1_testing_img],
    registered_list_dict={
        "rigid (feature-based)": [img1_registered['initial similarity'], img2_registered['initial similarity'], img1_test_registered['initial similarity']],
        "rigid (intensity-based)": [img1_registered['intensity based']['rigid'], img2_registered['intensity based']['rigid'], img1_test_registered['intensity based']['rigid']],
        "affine (intensity-based)": [img1_registered['intensity based']['affine'], img2_registered['intensity based']['affine'], img1_test_registered['intensity based']['affine']],
        "bspline (intensity-based)": [img1_registered['intensity based']['bspline'], img2_registered['intensity based']['bspline'], img1_test_registered['intensity based']['bspline']]
    }
)

#### ROI images

In [ ]:
import sys
sys.path.insert(0, "/home-link/zxovq55/Masters-Thesis/src/registration")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from tifffile import imread
from skimage.transform import resize
import numpy as np
import textwrap

import importlib, registration
importlib.reload(registration)
importlib.reload(registration.reg)
importlib.reload(registration.metrics)
importlib.reload(registration.regPipeline)
importlib.reload(registration.preprocess)

In [ ]:
hne_px_sz = 0.5023
dapi_px_sz = 0.209877

scale = hne_px_sz / dapi_px_sz

# load dapi
dapi_1_data = np.array(imread("../../data/dapi_1.tif"))
dapi1_init = resize(dapi_1_data, (int(dapi_1_data.shape[0]/scale), int(dapi_1_data.shape[1]/scale)), anti_aliasing=True)

dapi_2_data = np.array(imread("../../data/dapi_2.tif"))
dapi2_init = resize(dapi_2_data, (int(dapi_2_data.shape[0]/scale), int(dapi_2_data.shape[1]/scale)), anti_aliasing=True)

dapi_3_data = np.array(imread("../../data/dapi_3.tif"))
dapi3_init = resize(dapi_3_data, (int(dapi_3_data.shape[0]/scale), int(dapi_3_data.shape[1]/scale)), anti_aliasing=True)

dapi1_init, dapi2_init, dapi3_init = dapi1_init*255, dapi2_init*255, dapi3_init*255

# load hne
hne_1_init = np.array(imread("../../data/hne_1.tif"))
hne_2_init = np.array(imread("../../data/hne_2.tif"))
hne_3_init = np.array(imread("../../data/hne_3.tif"))

# preprocess hne
hne1_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne_1_init)
hne2_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne_2_init)
hne3_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne_3_init)

In [ ]:
def overlay_images(dapi_img, hne_img, axis, alpha_dapi=1, alpha_hne=0.35):
    axis.imshow(dapi_img, cmap='Greens', alpha=alpha_dapi)
    axis.imshow(hne_img, cmap='Reds', alpha=alpha_hne)
    axis.set_aspect('equal')
    axis.set_xticks([])
    axis.set_yticks([])

def get_nice_ticks(max_val, n_ticks=5):
    step = max_val // n_ticks
    step = int(np.ceil(step / 50) * 50)  # round to nearest 50
    ticks = np.arange(0, step * n_ticks, step)  # exclude the last value to avoid overlap
    return ticks

def plot_comparison_grid2(
        dapi_list,
        hne_list,
        deconv_list,
        registered_list_dict,
        column_titles=None
    ):
    n_rows = len(dapi_list)
    reg_methods = list(registered_list_dict.keys())
    n_cols = 3 + len(reg_methods)

    # set equal image sizes
    min_shape = np.min([img.shape[:2] for img in dapi_list], axis=0)
    def resize_to_min_shape(img_list, min_shape):
        return [img[:min_shape[0], :min_shape[1]] for img in img_list]

    dapi_list = resize_to_min_shape(dapi_list, min_shape)
    hne_list = resize_to_min_shape(hne_list, min_shape)
    deconv_list = resize_to_min_shape(deconv_list, min_shape)
    for method in reg_methods:
        registered_list_dict[method] = resize_to_min_shape(registered_list_dict[method], min_shape)

    if column_titles is None:
        column_titles = ["DAPI", "H&E", "initial overlay"] + reg_methods

    fig, axes = plt.subplots(
        n_rows + 1,
        n_cols,
        figsize=(1.5 * n_cols, 1.65 * (n_rows + 0.2)),
        squeeze=False
    )

    # top row titles
    for j in range(n_cols):
        ax = axes[0, j]
        title = column_titles[j]
        wrapped_title = "\n".join(textwrap.wrap(title, width=15))
        ax.set_title(wrapped_title, fontsize=12, pad=3)
        ax.axis("off")


    # image rows
    for i in range(n_rows):
        for j in range(n_cols):
            ax = axes[i + 1, j]
        
            # which image to plot
            if j == 0:
                img = dapi_list[i]
            elif j == 1:
                img = hne_list[i]
            elif j == 2:
                img = deconv_list[i]
            else:
                method = reg_methods[j-3]
                img = registered_list_dict[method][i]
        
            # overlay 
            if j >= 2:
                overlay_images(dapi_list[i], img, ax)
            else:
                ax.imshow(img, cmap="gray")
                ax.set_aspect('equal')
        
            # show only for left column (y) and bottom row (x)
            if j == 0:  # left column
                yticks = get_nice_ticks(min_shape[0], n_ticks=4)
                ax.set_yticks(yticks)
                ax.set_yticklabels([f"{int(y)}" for y in yticks], fontsize=8)
            else:
                ax.set_yticks([])

            if i == n_rows - 1:  # bottom row
                xticks = get_nice_ticks(min_shape[1], n_ticks=4)
                ax.set_xticks(xticks)
                ax.set_xticklabels([f"{int(x)}" for x in xticks], fontsize=8)
            else:
                ax.set_xticks([])


    plt.subplots_adjust(
        left=0.03, right=0.97,
        top=0.95, bottom=0.05,
        wspace=0.02,
        hspace=0.01
    )

    plt.show()

In [ ]:
img1_tforms, img1_registered, img1_tre_pts = registration.register_DAPI_HnE(dapi1_init, hne1_deconv, adv_tform='intensity', intensity_tform='r-af-bs', mpp=hne_px_sz)

In [ ]:
img2_tforms, img2_registered, img2_tre_pts = registration.register_DAPI_HnE(dapi2_init, hne2_deconv, adv_tform='intensity', intensity_tform='r-af-bs', mpp=hne_px_sz)

In [ ]:
img3_tforms, img3_registered, img3_tre_pts = registration.register_DAPI_HnE(dapi3_init, hne3_deconv, adv_tform='intensity', intensity_tform='r-af-bs', mpp=hne_px_sz)

In [ ]:
plot_comparison_grid2(
    dapi_list=[dapi1_init[:,1000:], np.rot90(dapi2_init, k=1), dapi3_init],
    hne_list=[hne_1_init[:,1000:], np.rot90(hne_2_init, k=1), hne_3_init],
    deconv_list=[hne1_deconv[:,1000:], np.rot90(hne2_deconv, k=1), hne3_deconv],
    registered_list_dict={
        "rigid (feature-based)": [img1_registered['initial similarity'][:,1000:], np.rot90(img2_registered['initial similarity'], k=1), img3_registered['initial similarity']],
        "rigid (intensity-based)": [img1_registered['intensity based']['rigid'][:,1000:], np.rot90(img2_registered['intensity based']['rigid'], k=1), img3_registered['intensity based']['rigid']],
        "affine (intensity-based)": [img1_registered['intensity based']['affine'][:,1000:], np.rot90(img2_registered['intensity based']['affine'], k=1), img3_registered['intensity based']['affine']],
        "bspline (intensity-based)": [img1_registered['intensity based']['bspline'][:,1000:], np.rot90(img2_registered['intensity based']['bspline'], k=1), img3_registered['intensity based']['bspline']]
    }
)

### plots

##### Comparison between methods

In [ ]:
# Boxplot for different registration methods
# rTRE

num_imgs = 3
tre_metrics = []
before_list_all = []

for file_tag in ['ftr', 'r', 'af', 'bs']:
    metrics_list = []
    
    for img_idx in range(num_imgs):

        with open(f"result_plots_metrics/{img_idx}_tre_{file_tag}_dict.pkl", "rb") as f:
            tre_dict = pickle.load(f)

            metrics_list = np.concatenate((metrics_list, tre_dict[(f"img_{img_idx+1}")]), axis=0)
            metrics_list = metrics_list.ravel()

    tre_metrics.append(metrics_list)

# load before registration values
for img_idx in range(num_imgs):

    with open(f"result_plots_metrics/{img_idx}_tre_bg_dict.pkl", "rb") as f:
        bg_dict = pickle.load(f)
        before_values = bg_dict[f"img_{img_idx+1}"]
        before_list_all.append(before_values)

before_median = np.median(np.concatenate(before_list_all))

methods = ["rigid\n(feature-based)", "rigid\n(intensity-based)", "affine\n(intensity-based)", "bspline\n(intensity-based)"]

# scaling to 10^-4 
tre_metrics = [arr * 1e4 for arr in tre_metrics]
before_median = before_median * 1e4


fig, ax = plt.subplots(figsize=(8,6))

box = ax.boxplot(tre_metrics, tick_labels=methods, patch_artist=True, showfliers=True, 
                 capprops=dict(color='none'), 
                 whiskerprops=dict(color='black'), 
                 boxprops=dict(color='black'), 
                 medianprops=dict(color='black', linewidth=2),
                 flierprops=dict(marker='o', markersize=6, markerfacecolor='black'))

colors = ["lightblue", "lightgreen", "lightpink", "lightyellow"]
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)

y_min = min(np.min(arr) for arr in tre_metrics)
y_max = max(np.max(arr) for arr in tre_metrics)

ax.set_ylim(y_min-0.5, y_max+3)

# add before registration median as text box
ax.text(
    2.5,
    y_max + 1.2,
    f"Before-registration median: {before_median:.0f}",
    ha="center",
    va="bottom",
    fontsize=12,
    bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black")
)

ax.set_ylabel("rTRE (×10⁻⁴)", fontsize=16)
ax.set_title("Comparison of Registration Methods", fontsize=20)
ax.tick_params(axis='x', labelsize=11)
plt.show()


In [ ]:
# wilcoxon signed-rank test for rTRE

num_imgs = 3

ftr_tre_medians = []
r_tre_medians = []

for img_idx in range(num_imgs):

    with open(f"result_plots_metrics/{img_idx}_tre_ftr_dict.pkl", "rb") as f:
        d = pickle.load(f)
        arr = d[f"img_{img_idx+1}"]
        ftr_tre_medians.append(np.median(arr))

    with open(f"result_plots_metrics/{img_idx}_tre_r_dict.pkl", "rb") as f:
        d = pickle.load(f)
        arr = d[f"img_{img_idx+1}"]
        r_tre_medians.append(np.median(arr))

stat, p_value = wilcoxon(ftr_tre_medians, r_tre_medians, alternative='two-sided')

print("--------------------Wilcoxon signed-rank test--------------------")
print("Statistic:", stat)
print("p-value:", p_value)

#############################################################################

# Mann-Whitney U test/ Wilcoxon rank-sum test for rTRE

rigid_tre_feature = tre_metrics[0]  
rigid_tre_intensity = tre_metrics[1]

stat, p_value = mannwhitneyu(rigid_tre_feature, rigid_tre_intensity, alternative='two-sided')

print("\n--------------------Mann–Whitney U test--------------------")
print("U-statistic:", stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("Difference between rigid feature-based and rigid intensity-based is statistically significant.")
else:
    print("Difference is not statistically significant.")

In [ ]:
# Boxplot for different registration methods
# MI

import numpy as np
import matplotlib.pyplot as plt
import pickle

num_imgs = 3
mi_metrics = []
before_list_all = []  

for file_tag in ['ftr', 'r', 'af', 'bs']:
    metrics_list = []

    for img_idx in range(num_imgs):

        # after registration metrics
        with open(f"result_plots_metrics/{img_idx}_mi_{file_tag}_dict.pkl", "rb") as f:
            mi_dict = pickle.load(f)
            arr = mi_dict[(f"img_{img_idx+1}")]
            metrics_list = np.concatenate((metrics_list, arr), axis=0)
            metrics_list = metrics_list.ravel()

    mi_metrics.append(metrics_list)



# load before registration values
for img_idx in range(num_imgs):

    with open(f"result_plots_metrics/{img_idx}_mi_bg_dict.pkl", "rb") as f:
        bg_dict = pickle.load(f)
        before_values = bg_dict[f"img_{img_idx+1}"]  
        before_list_all.append(before_values)

before_median = np.median(np.concatenate(before_list_all))


methods = ["rigid\n(feature-based)", "rigid\n(intensity-based)",
           "affine\n(intensity-based)", "bspline\n(intensity-based)"]


# scaling to 10^-2 
mi_metrics = [arr * 1e2 for arr in mi_metrics]
before_median = before_median * 1e2


fig, ax = plt.subplots(figsize=(8,6))

box = ax.boxplot(mi_metrics, tick_labels=methods, patch_artist=True, showfliers=True,
                 capprops=dict(color='none'),
                 whiskerprops=dict(color='black'),
                 boxprops=dict(color='black'),
                 medianprops=dict(color='black', linewidth=2),
                 flierprops=dict(marker='o', markersize=6, markerfacecolor='black'))

# color the boxes
colors = ["lightblue", "lightgreen", "lightpink", "lightyellow"]
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)

y_min = min(np.min(arr) for arr in mi_metrics)
y_max = max(np.max(arr) for arr in mi_metrics)
ax.set_ylim(y_min - 0.5, y_max + 3.5)

# add before registration median as text box
ax.text(
    2.5,                   # middle of the x-axis
    y_max + 1.5,          # above the boxplots
    f"Before-registration median: {before_median:.1f}",
    ha="center",
    va="bottom",
    fontsize=12,
    bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black")
)

ax.set_ylabel("NMI (×10⁻²)", fontsize=16)
ax.set_title("Comparison of Registration Methods", fontsize=20)
ax.tick_params(axis='x', labelsize=11)

plt.show()


In [ ]:
# wilcoxon signed-rank test for MI

num_imgs = 3

ftr_mi_medians = []
r_mi_medians = []

for img_idx in range(num_imgs):

    with open(f"result_plots_metrics/{img_idx}_mi_ftr_dict.pkl", "rb") as f:
        d = pickle.load(f)
        arr = d[f"img_{img_idx+1}"]
        ftr_mi_medians.append(np.median(arr))

    with open(f"result_plots_metrics/{img_idx}_mi_r_dict.pkl", "rb") as f:
        d = pickle.load(f)
        arr = d[f"img_{img_idx+1}"]
        r_mi_medians.append(np.median(arr))

stat, p_value = wilcoxon(ftr_mi_medians, r_mi_medians, alternative='two-sided')

print("--------------------Wilcoxon signed-rank test--------------------")
print("Statistic:", stat)
print("p-value:", p_value)

##########################################################################################

# Mann-Whitney U test/ Wilcoxon rank-sum test for MI

rigid_mi_feature = mi_metrics[0]  
rigid_mi_intensity = mi_metrics[1]

stat, p_value = mannwhitneyu(rigid_mi_feature, rigid_mi_intensity, alternative='two-sided')

print("\n--------------------Mann–Whitney U test--------------------")
print("U-statistic:", stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("Difference between rigid feature-based and rigid intensity-based is statistically significant.")
else:
    print("Difference is not statistically significant.")


In [ ]:
# Scatter plot comparing two methods 
# tre

ftr_r = tre_metrics[0] # rigid (feature-based)
intns = tre_metrics[1] # rigid (intensity-based)

fig, ax = plt.subplots(figsize=(8,8))

ax.scatter(ftr_r, intns, c='black', s=60, alpha=0.8, edgecolor='k')
min_val = min(ftr_r.min(), intns.min())
max_val = max(ftr_r.max(), intns.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5)

ax.set_xlabel("rigid (feature-based) (×10⁻⁴)", fontsize=16)
ax.set_ylabel("rigid (intensity-based) (×10⁻⁴)", fontsize=16)
ax.set_title("Pairwise Comparison of Two Methods with rTRE", fontsize=20)
ax.set_aspect('equal', adjustable='box')  


plt.show()

In [ ]:
# Scatter plot comparing two methods 
# mi

ftr_r = mi_metrics[0] # rigid (feature-based)
intns = mi_metrics[1] # rigid (intensity-based)

fig, ax = plt.subplots(figsize=(8,8))

ax.scatter(ftr_r, intns, c='black', s=60, alpha=0.8, edgecolor='k')
min_val = min(ftr_r.min(), intns.min())
max_val = max(ftr_r.max(), intns.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5)

ax.invert_yaxis()
ax.invert_xaxis()
ax.set_xlabel("rigid (feature-based) (×10⁻²)", fontsize=16)
ax.set_ylabel("rigid (intensity-based) (×10⁻²)", fontsize=16)
ax.set_title("Pairwise Comparison of Two Methods with NMI", fontsize=20)
ax.set_aspect('equal', adjustable='box')  


plt.show()

##### Comparison between runs 

In [ ]:
num_imgs = 3

# Colors
color_ftr = "skyblue"
color_r   = "lightgreen"  
scatter_ftr = "blue"
scatter_r = "darkgreen"

ftr_values = []
r_values = []

# Load data
for img_idx in range(num_imgs):
    # ftr
    with open(f"result_plots_metrics/{img_idx}_tre_ftr_dict.pkl", "rb") as f:
        tre_dict = pickle.load(f)
    vals_ftr = np.asarray(tre_dict[f"img_{img_idx+1}"]).ravel()
    ftr_values.append(vals_ftr)

    # r
    with open(f"result_plots_metrics/{img_idx}_tre_r_dict.pkl", "rb") as f:
        tre_dict = pickle.load(f)
    vals_r = np.asarray(tre_dict[f"img_{img_idx+1}"]).ravel()
    r_values.append(vals_r)

r_values = [value * 1e4 for value in r_values]
ftr_values = [value * 1e4 for value in ftr_values]

fig, ax = plt.subplots(figsize=(10, 6))
positions = np.arange(num_imgs)

# plot violins and scatter points
for i in range(num_imgs):
    for vals, pos, color, scatter_color in zip([ftr_values[i], r_values[i]],
                                               [positions[i]-0.15, positions[i]+0.15],
                                               [color_ftr, color_r],
                                               [scatter_ftr, scatter_r]):
        vp = ax.violinplot(
            vals,
            positions=[pos],
            widths=0.5,
            showmeans=False,
            showmedians=True
        )
        for part in vp['bodies']:
            part.set_facecolor(color)
            part.set_edgecolor('black')
            part.set_alpha(0.7)
        vp['cmedians'].set_color('black')

        # points for all runs
        jitter = np.random.uniform(-0.07, 0.07, size=len(vals))
        ax.scatter(pos + jitter, vals, color=scatter_color, s=30, alpha=0.8)


ax.set_xticks(positions)
ax.set_xticklabels([f"Image {i+1}" for i in range(num_imgs)], fontsize=16)
ax.set_ylabel("rTRE (×10⁻⁴)", fontsize=18)
ax.set_title("Comparison of rTRE Distributions at different runs", fontsize=20)

# legend
ax.plot([], [], color=color_ftr, label="Feature-based Rigid", linewidth=10)
ax.plot([], [], color=color_r, label="Intensity-based Rigid", linewidth=10)
ax.scatter([], [], color=scatter_color, s=30, label="Individual runs")
ax.legend(fontsize=16)

plt.tight_layout()
plt.show()


In [ ]:
num_imgs = 3

color_ftr = "skyblue"
color_r   = "lightgreen"  
scatter_ftr = "blue"
scatter_r = "darkgreen"

ftr_values = []
r_values = []

# load NMI data
for img_idx in range(num_imgs):
    # ftr
    with open(f"result_plots_metrics/{img_idx}_mi_ftr_dict.pkl", "rb") as f:
        nmi_dict = pickle.load(f)
    vals_ftr = np.asarray(nmi_dict[f"img_{img_idx+1}"]).ravel()
    ftr_values.append(vals_ftr)

    # r
    with open(f"result_plots_metrics/{img_idx}_mi_r_dict.pkl", "rb") as f:
        nmi_dict = pickle.load(f)
    vals_r = np.asarray(nmi_dict[f"img_{img_idx+1}"]).ravel()
    r_values.append(vals_r)


r_values = [value * 1e2 for value in r_values]
ftr_values = [value * 1e2 for value in ftr_values]

fig, ax = plt.subplots(figsize=(10, 6))
positions = np.arange(num_imgs)

# plot violins + scatter points
for i in range(num_imgs):
    for vals, pos, color, scatter_color in zip([ftr_values[i], r_values[i]],
                                               [positions[i]-0.15, positions[i]+0.15],
                                               [color_ftr, color_r],
                                               [scatter_ftr, scatter_r]):
        vp = ax.violinplot(
            vals,
            positions=[pos],
            widths=0.5,
            showmeans=False,
            showmedians=True
        )
        for part in vp['bodies']:
            part.set_facecolor(color)
            part.set_edgecolor('black')
            part.set_alpha(0.7)
        vp['cmedians'].set_color('black')

        # scatter points for all runs
        jitter = np.random.uniform(-0.07, 0.07, size=len(vals))
        ax.scatter(pos + jitter, vals, color=scatter_color, s=30, alpha=0.8)

ax.set_xticks(positions)
ax.set_xticklabels([f"Image {i+1}" for i in range(num_imgs)], fontsize=16)
ax.set_ylabel("NMI (×10⁻²)", fontsize=18)
ax.set_title("Comparison of NMI Distributions at Different Runs", fontsize=20)

# legend
ax.plot([], [], color=color_ftr, label="Feature-based Rigid", linewidth=10)
ax.plot([], [], color=color_r, label="Intensity-based Rigid", linewidth=10)
ax.scatter([], [], color=scatter_color, s=30, label="Individual runs")
ax.legend(fontsize=16)

plt.tight_layout()
plt.show()


##### Comparison between resolution and SIFT parameters

In [ ]:
# removing boxes with None values from the boxplots


# parameter
n_octaves = [2, 3, 4]
n_scales = [2, 3, 4, 5]
res_levels = [2, 4, 8, 12, 16]

with open("result_plots_metrics/tre_dict.pkl", "rb") as f:
    metric = pickle.load(f)

# scale to 10^-4
metric = {k : [vv * 1e4 for vv in v if vv is not None] for k, v in metric.items()}

row_y_limits = []
for o in n_octaves:
    all_vals = []
    for s in n_scales:
        for r in res_levels:
            vals = metric[(o, s, r)]
            if vals is None:
                continue
            # remove boxes with any None
            if any(v is None for v in vals):
                continue
            all_vals.extend(vals)


    if len(all_vals) == 0:
        row_y_limits.append((0, 1))
    else:
        min_val = min(all_vals)
        max_val = max(all_vals)
        buffer = (max_val - min_val) * 0.5
        row_y_limits.append((min_val - buffer, max_val + buffer))



# boxplot function 
def plot_box_median(ax, data_dict, res_levels, norm, cmap,
                    show_xticks=False, show_yticks=False, y_lim=None):

    positions = np.arange(1, len(res_levels) + 1)
    box_width = 0.45

    # each box 
    for i, r in enumerate(res_levels):
        vals = data_dict[r]
        if vals is None:
            continue
        # remove any box containing None
        if any(v is None for v in vals):
            continue

        med = np.median(vals)
        bp = ax.boxplot(
            [vals],
            positions=[positions[i]],
            widths=box_width,
            patch_artist=True,
            showfliers=False,
            boxprops=dict(color="black"),
            whiskerprops=dict(color="black"),
            capprops=dict(color="none"),
            medianprops=dict(color="black", linewidth=1.5)
        )
        bp["boxes"][0].set_facecolor(cmap(norm(med)))

    ax.set_xticks(positions)

    if show_xticks:
        ax.set_xticklabels(res_levels, fontsize=9)
        ax.spines['bottom'].set_visible(True)
        ax.tick_params(axis='x', which='both', bottom=True, top=False)
    else:
        ax.set_xticklabels([])
        ax.spines['bottom'].set_visible(False)
        ax.tick_params(axis='x', which='both', bottom=False, top=False)

    if show_yticks:
        ax.spines['left'].set_visible(True)
        ax.set_ylabel("rTRE (×10⁻⁴)", fontsize=10)
        if y_lim is not None:
            ax.set_ylim(y_lim)
            ticks = np.linspace(y_lim[0], y_lim[1], 4)
            ax.set_yticklabels([f"{t:.0f}" for t in ticks], fontsize=9)
    else:
        ax.set_yticks([])
        ax.set_yticklabels([])
        ax.spines['left'].set_visible(False)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


# colormap normalization 
all_medians = []
for o in n_octaves:
    for s in n_scales:
        for r in res_levels:
            vals = metric[(o, s, r)]
            if vals is None:
                continue
            if any(v is None for v in vals):
                continue
            all_medians.append(np.median(vals))

if len(all_medians) == 0:
    raise RuntimeError("No valid boxes found (all contain None).")

norm = colors.Normalize(vmin=min(all_medians), vmax=max(all_medians))
cmap = cm.viridis

fig, axes = plt.subplots(
    len(n_octaves),
    len(n_scales),
    figsize=(2.6 * len(n_scales), 1.8 * len(n_octaves)),
    sharex=False,
    sharey=False
)

axes = np.array(axes)[::-1, :]
label_box = dict(boxstyle="round,pad=0.3", fc="white", ec="black", lw=1)

for row_idx, o in enumerate(n_octaves):
    y_lim = row_y_limits[row_idx]
    is_bottom_row = (row_idx == 0)

    for col_idx, s in enumerate(n_scales):
        ax = axes[row_idx, col_idx]
        subset = {r: metric[(o, s, r)] for r in res_levels}

        plot_box_median(
            ax,
            subset,
            res_levels,
            norm,
            cmap,
            show_xticks=is_bottom_row,
            show_yticks=(col_idx == 0),
            y_lim=y_lim
        )

        if row_idx == len(n_octaves) - 1:
            ax.set_title(f"      n_scales = {s}      ", fontsize=11, bbox=label_box)

        if is_bottom_row:
            ax.set_xlabel("Resolution", fontsize=10)

# n_octaves labels
x_pos = 0.06
y_shift = 0.05
compression = 0.78

for row_idx, o in enumerate(n_octaves):
    y_center = (row_idx + 0.6) / len(n_octaves)
    y_center = 0.5 + (y_center - 0.5) * compression + y_shift

    fig.text(
        x_pos, y_center,
        f"n_octaves = {o}",
        fontsize=10,
        rotation=90,
        va="center", ha="center",
        bbox=label_box
    )

# colorbar
cbar_ax = fig.add_axes([0.94, 0.15, 0.015, 0.7])
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('rTRE median', fontsize=12)

fig.subplots_adjust(left=0.16, right=0.92, top=0.92, bottom=0.22, hspace=0.35)
plt.show()


In [ ]:
# removing boxes with <4 non-None values from the boxplots

# parameters
n_octaves = [2, 3, 4]
n_scales = [2, 3, 4, 5]
res_levels = [2, 4, 8, 12, 16]

with open("result_plots_metrics/tre_dict.pkl", "rb") as f:
    metric = pickle.load(f)

# scale to 10^-4
# scale to 10^-4
metric = {k : [vv * 1e4 for vv in v if vv is not None] for k, v in metric.items()}

row_y_limits = []
for o in n_octaves:
    all_vals = []
    for s in n_scales:
        for r in res_levels:
            vals = metric[(o, s, r)]
            if vals is None:
                continue
            clean_vals = [v for v in vals if v is not None]
            if len(clean_vals) < 4:
                continue
            all_vals.extend(clean_vals)

    if len(all_vals) == 0:
        row_y_limits.append((0, 1))
    else:
        min_val = min(all_vals)
        max_val = max(all_vals)
        buffer = (max_val - min_val) * 0.5
        row_y_limits.append((min_val - buffer, max_val + buffer))



# boxplot function 
def plot_box_median(ax, data_dict, res_levels, norm, cmap,
                    show_xticks=False, show_yticks=False, y_lim=None):

    positions = np.arange(1, len(res_levels) + 1)

    box_width = 0.45
    for i, r in enumerate(res_levels):
        vals = data_dict[r]
        if vals is None:
            continue
        clean_vals = [v for v in vals if v is not None]
        if len(clean_vals) < 4:
            continue

        med = np.median(clean_vals)
        bp = ax.boxplot(
            [clean_vals],
            positions=[positions[i]],
            widths=box_width,
            patch_artist=True,
            showfliers=False,
            boxprops=dict(color="black"),
            whiskerprops=dict(color="black"),
            capprops=dict(color="none"),
            medianprops=dict(color="black", linewidth=1.5)
        )
        bp["boxes"][0].set_facecolor(cmap(norm(med)))

    ax.set_xticks(positions)

    # show labels only on bottom row
    if show_xticks:
        ax.set_xticklabels(res_levels, fontsize=9)
        ax.spines['bottom'].set_visible(True)
        ax.tick_params(axis='x', which='both', bottom=True, top=False)
    else:
        ax.set_xticklabels([])   # hide labels
        ax.spines['bottom'].set_visible(False)
        ax.tick_params(axis='x', which='both', bottom=False, top=False)  # hide tick lines


    if show_yticks:
        ax.spines['left'].set_visible(True)
        ax.set_ylabel("rTRE (×10⁻⁴)", fontsize=10)
        if y_lim is not None:
            ax.set_ylim(y_lim)
            ticks = np.linspace(y_lim[0], y_lim[1], 4)
            ax.set_yticklabels([f"{t:.0f}" for t in ticks], fontsize=9)
    else:
        ax.set_yticks([])
        ax.set_yticklabels([])
        ax.spines['left'].set_visible(False)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


# colormap normalization 
all_medians = []
for o in n_octaves:
    for s in n_scales:
        for r in res_levels:
            vals = metric[(o, s, r)]
            if vals is None:
                continue
            clean_vals = [v for v in vals if v is not None]
            if len(clean_vals) < 4:
                continue
            all_medians.append(np.median(clean_vals))

if len(all_medians) == 0:
    raise RuntimeError("No valid boxes found (need at least one cell with >=4 non-None values).")

norm = colors.Normalize(vmin=min(all_medians), vmax=max(all_medians))
cmap = cm.viridis

fig, axes = plt.subplots(
    len(n_octaves),
    len(n_scales),
    figsize=(2.6 * len(n_scales), 1.8 * len(n_octaves)),
    sharex=False,
    sharey=False
)

axes = np.array(axes)[::-1, :]
label_box = dict(boxstyle="round,pad=0.3", fc="white", ec="black", lw=1)

for row_idx, o in enumerate(n_octaves):
    y_lim = row_y_limits[row_idx]
    is_bottom_row = (row_idx == 0)

    for col_idx, s in enumerate(n_scales):
        ax = axes[row_idx, col_idx]
        subset = {r: metric[(o, s, r)] for r in res_levels}

        plot_box_median(
            ax,
            subset,
            res_levels,
            norm,
            cmap,
            show_xticks=is_bottom_row,
            show_yticks=(col_idx == 0),
            y_lim=y_lim
        )

        # column label 
        if row_idx == len(n_octaves) - 1:
            ax.set_title(f"      n_scales = {s}      ", fontsize=11, bbox=label_box)

        if is_bottom_row:
            ax.set_xlabel("Resolution", fontsize=10)

# n_octaves labels
x_pos = 0.06
y_shift = 0.05
compression = 0.78

for row_idx, o in enumerate(n_octaves):
    y_center = (row_idx + 0.6) / len(n_octaves)
    y_center = 0.5 + (y_center - 0.5) * compression + y_shift

    fig.text(
        x_pos, y_center,
        f"n_octaves = {o}",
        fontsize=10,
        rotation=90,
        va="center", ha="center",
        bbox=label_box
    )

# colorbar
cbar_ax = fig.add_axes([0.94, 0.15, 0.015, 0.7])
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('rTRE median', fontsize=12)


fig.subplots_adjust(left=0.16, right=0.92, top=0.92, bottom=0.22, hspace=0.35)
plt.show()
